# Attention encoder with per-cell presence supervision
Use the existing 128-wide attention encoder with entmax-gated conditional AD and frozen LRN_AD. Training minimizes reconstruction plus per-cell BCE only; there is no distinct-element sparsity penalty.

$$\hat A_{l,e}=p^{\mathrm{entmax}}_{l,e}\,\mathrm{softplus}(r_{l,e}),\qquad L=L_{\mathrm{reconstruction}}+\lambda L_{\mathrm{BCE}}.$$
BCEWithLogits supervises the absent/present score difference. Entmax remains the AD gate; its reported probabilities are not the sigmoid probabilities corresponding to BCE and require separate calibration assessment.

`resume=True` continues checkpoints in OUTPUT_DIR, falling back to protected best. All training settings are at the top of the training cell. Existing checkpoints can resume with the revised objective. Validation reconstruction still selects protected best weights.


In [1]:
from pathlib import Path
import sys, re, json, copy, math, hashlib
try:
    import tomllib
except ImportError:
    import tomli as tomllib
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ibeamlab/ibeamnet.py').exists())
sys.path.insert(0, str(ROOT))
EXAMPLES = ROOT / 'examples'
import ibeamlab as ibl
from ibeamlab import LRN_AD
import importlib
import ibeamlab.spectrum_encoder as encoder_module
importlib.reload(encoder_module)
SpectrumEntmaxEncoder = encoder_module.SpectrumEntmaxEncoder
from ibeamlab._native import native
import time

binary_entmax15_loss = encoder_module.binary_entmax15_loss



c:\Users\local_admin\.conda\envs\DL\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Dataset and frozen decoder


In [2]:
DATASET_ROOT = EXAMPLES / 'datasets'
CONFIG_PATH = EXAMPLES / 'multilayer-generation.toml'
LRN_AD_CHECKPOINT = EXAMPLES / 'artifacts/lrn_ad_rbs_10layers/lrn_ad_state.pt'
# Both stored spectra and LRN_AD targets must be pileup-free expected counts.
# For noisy datasets, switch off Poisson augmentation; the clean-target assumption then differs.
SEED = 7
CPU_THREADS = 8
torch.set_num_threads(CPU_THREADS)
if not LRN_AD_CHECKPOINT.is_file():
    raise FileNotFoundError(f'Choose the frozen LRN_AD checkpoint: {LRN_AD_CHECKPOINT}')
lrn_checkpoint = torch.load(LRN_AD_CHECKPOINT, map_location='cpu', weights_only=False)
MAX_LAYERS = int(lrn_checkpoint['max_layers'])
POISSON_NOISE = True
VALIDATION_SAMPLES = 20_000
TEST_SAMPLES = 20_000
METHOD = lrn_checkpoint['method']
LRN_AD_OUTPUT_SCALE = float(lrn_checkpoint['output_scale'])
LRN_AD_INPUT_FACTOR = float(lrn_checkpoint['ad_input_factor'])
if not np.isfinite(LRN_AD_INPUT_FACTOR) or LRN_AD_INPUT_FACTOR <= 0:
    raise ValueError('Invalid LRN_AD input factor')
LRN_AD_ARCHITECTURE = dict(lrn_checkpoint['architecture'])
if not np.isfinite(LRN_AD_OUTPUT_SCALE) or LRN_AD_OUTPUT_SCALE <= 0: raise ValueError('Invalid LRN_AD output scale')



## 2. Data and reproducible splits

Targets are clean spectra. If the Poisson augmentation is enabled, train on newly drawn counts and evaluate with fixed noise draws. All dataset/configuration checks are retained.


In [3]:
configuration = ibl.load_generation_configuration(CONFIG_PATH)
if configuration.variations:
    raise ValueError("Encoder requires fixed beam/detector settings; choose a config without setup variations")
if not 1 <= MAX_LAYERS <= configuration.maximum_layers:
    raise ValueError("MAX_LAYERS exceeds the configuration")
study = configuration.study(MAX_LAYERS)
ELEMENTS = tuple(configuration.elements)
LABELS = tuple(d.label for d in study.experiment.detectors)
LAYER_DATASETS = {}
for path in sorted(DATASET_ROOT.rglob("layers_*")):
    match = re.fullmatch(r"layers_(\d+)", path.name)
    if path.is_dir() and match and 1 <= int(match.group(1)) <= MAX_LAYERS:
        LAYER_DATASETS.setdefault(int(match.group(1)), []).append(path)
if not LAYER_DATASETS:
    raise FileNotFoundError(f"No layers_XX datasets under {DATASET_ROOT}")
print("Detector order:", LABELS)
print("Element order:", ELEMENTS)
print("Maximum layers:", MAX_LAYERS)
if lrn_checkpoint['max_layers'] != MAX_LAYERS or METHOD not in LABELS:
    raise ValueError('LRN_AD method/layer count differs')
if tuple(lrn_checkpoint['input_parameter_names']) != tuple(study.parameter_names):
    raise ValueError('LRN_AD physical parameter order differs')
if lrn_checkpoint.get('layer_representation') != 'areal_density':
    raise ValueError('Use a checkpoint from train_lrn_ad_multilayer.ipynb, not the physical-parameter LRN')
if tuple(lrn_checkpoint.get('elements', ())) != ELEMENTS:
    raise ValueError('LRN_AD element order differs from the encoder configuration')
if len(LABELS) != 1:
    raise ValueError('This first notebook supports the single-detector LRN_AD notebook schema')
actual_datasets = [str(p.resolve()) for paths in LAYER_DATASETS.values() for p in paths]
dataset_manifest = [dict(path=str(path.resolve()),
    files=[(file.name, file.stat().st_size, file.stat().st_mtime_ns)
           for file in sorted(path.iterdir()) if file.is_file()])
    for paths in LAYER_DATASETS.values() for path in paths]


Detector order: ('RBS',)
Element order: ('Li', 'Ni', 'Mn', 'Co', 'O', 'C', 'F', 'P', 'Cu', 'Al')
Maximum layers: 10


In [4]:
def template_table(text):
    value = tomllib.loads(text)
    value.pop("format", None)
    value.pop("format_version", None)
    return value

def load_case(dataset_path, layer_count):
    dataset = ibl.open_dataset(dataset_path)
    if not dataset.metadata.complete or not len(dataset):
        raise ValueError(f"Incomplete or empty dataset: {dataset_path}")
    case = configuration.study(layer_count)
    config = case._config()
    provenance = tomllib.loads(dataset._reader.metadata.generation_config_toml)
    expected_setup = template_table(native.sample.setup_to_toml(config.setup))
    expected_sample = template_table(native.sample.sample_to_toml(config.sample))
    if provenance.get("setup") != expected_setup or provenance.get("sample") != expected_sample:
        raise ValueError(f"Dataset template/setup differs from CONFIG_PATH: {dataset_path}")
    expected_methods = [{"label": m.label, "iba_method": m.iba_method,
                         "reference_file": Path(m.reference_file).as_posix()} for m in config.methods]
    if provenance.get("methods") != expected_methods:
        raise ValueError(f"Dataset method/reference differs from CONFIG_PATH: {dataset_path}")
    names = tuple(dataset.metadata.parameter_names)
    if len(set(names)) != len(names) or set(names) != set(case.parameter_names):
        raise ValueError(f"Parameter names do not match the case study: {dataset_path}")
    stored_targets = {p["name"]: p["target"] for p in provenance.get("parameters", [])}
    for p in case.parameters:
        expected_target = ({"type": "layer_thickness", "layer": p.layer} if p.kind == "layer_thickness"
                           else {"type": "species_concentration", "layer": p.layer, "element": p.element})
        if stored_targets.get(p.name) != expected_target:
            raise ValueError(f"Parameter target differs for {p.name}: {dataset_path}")
    rows = np.asarray(dataset.parameters, dtype=np.float32)
    columns = {name: i for i, name in enumerate(names)}
    densities = np.zeros((len(rows), MAX_LAYERS, len(ELEMENTS)), np.float32)
    for layer_index in range(layer_count):
        layer = case.experiment.sample.layers[layer_index]
        thickness = np.full(len(rows), layer.thickness, np.float32)
        fractions = np.tile([next(s.concentration for s in layer.species if s.element == element)
                             for element in ELEMENTS], (len(rows), 1)).astype(np.float32)
        for p in case.parameters:
            if p.layer != layer_index:
                continue
            if p.kind == "layer_thickness":
                thickness = rows[:, columns[p.name]]
            elif p.kind == "concentration":
                fractions[:, ELEMENTS.index(p.element)] = rows[:, columns[p.name]]
        if np.any(thickness < 0) or np.any(fractions < 0) or not np.allclose(fractions.sum(1), 1, atol=1e-6, rtol=0):
            raise ValueError(f"Invalid thickness/composition: {dataset_path}")
        densities[:, layer_index] = thickness[:, None] * fractions
    spectra = {label: np.asarray(dataset.spectra(label), dtype=np.float32) for label in LABELS}
    if any(not np.isfinite(x).all() or np.any(x < 0) for x in spectra.values()) or not np.isfinite(densities).all():
        raise ValueError(f"Nonfinite or negative training data: {dataset_path}")
    return spectra, densities

loaded = []
SPECTRUM_LENGTHS = {label: 0 for label in LABELS}
for layer_count, paths in sorted(LAYER_DATASETS.items()):
    for path in paths:
        spectra, densities = load_case(path, layer_count)
        for label in LABELS:
            SPECTRUM_LENGTHS[label] = max(SPECTRUM_LENGTHS[label], spectra[label].shape[1])
        loaded.append((spectra, densities))
        print(f"Loaded {path}: {len(densities):,} samples, {layer_count} layers")
x_parts, a_parts = [], []
for spectra, densities in loaded:
    x_parts.append(np.concatenate([
        np.pad(spectra[label], ((0, 0), (0, SPECTRUM_LENGTHS[label] - spectra[label].shape[1])))
        for label in LABELS], axis=1))
    a_parts.append(densities)
x_raw = np.concatenate(x_parts)
a_raw = np.concatenate(a_parts)
del loaded, x_parts, a_parts
print("Raw spectra:", x_raw.shape, "EDP:", a_raw.shape)

Loaded d:\ibeamlab\examples\datasets\iba_multilayer_moreC_seed3\layers_01: 200,000 samples, 1 layers
Loaded d:\ibeamlab\examples\datasets\iba_multilayer_seed_1\layers_01: 200,000 samples, 1 layers
Loaded d:\ibeamlab\examples\datasets\iba_multilayer_seed_2\layers_01: 200,000 samples, 1 layers
Loaded d:\ibeamlab\examples\datasets\iba_multilayer_moreC_seed3\layers_02: 100,000 samples, 2 layers
Loaded d:\ibeamlab\examples\datasets\iba_multilayer_seed_1\layers_02: 100,000 samples, 2 layers
Loaded d:\ibeamlab\examples\datasets\iba_multilayer_seed_2\layers_02: 100,000 samples, 2 layers
Loaded d:\ibeamlab\examples\datasets\iba_multilayer_moreC_seed3\layers_03: 100,000 samples, 3 layers
Loaded d:\ibeamlab\examples\datasets\iba_multilayer_seed_1\layers_03: 100,000 samples, 3 layers
Loaded d:\ibeamlab\examples\datasets\iba_multilayer_seed_2\layers_03: 100,000 samples, 3 layers
Loaded d:\ibeamlab\examples\datasets\iba_multilayer_moreC_seed3\layers_04: 100,000 samples, 4 layers
Loaded d:\ibeamlab\e

In [5]:
expected_width = lrn_checkpoint['model_state_dict']['output_scale'].numel()
if SPECTRUM_LENGTHS[METHOD] != expected_width:
    raise ValueError('Dataset channel width differs from the frozen decoder')
if len(x_raw) < 3:
    raise ValueError('At least three records are needed for train/validation/test')
for count in (VALIDATION_SAMPLES, TEST_SAMPLES):
    if not isinstance(count, int) or isinstance(count, bool) or count < 1:
        raise ValueError('Validation/test counts must be positive integers')
# Cap held-out sets at 10% each for small datasets, and at 20k by default.
val_count = min(VALIDATION_SAMPLES, max(1, len(x_raw)//10))
test_count = min(TEST_SAMPLES, max(1, len(x_raw)//10))
indices = np.random.default_rng(SEED).permutation(len(x_raw))
train_indices = indices[:len(indices)-val_count-test_count]
val_indices = indices[len(train_indices):len(train_indices)+val_count]
test_indices = indices[len(train_indices)+val_count:]

def fixed_poisson(expected, seed):
    generator = torch.Generator().manual_seed(seed)
    result = np.empty_like(expected)
    for start in range(0, len(expected), 512):
        result[start:start+512] = torch.poisson(torch.from_numpy(expected[start:start+512]), generator=generator).numpy()
    return result
x_train = x_raw[train_indices]
x_val_clean, x_test_clean = x_raw[val_indices], x_raw[test_indices]
x_val = fixed_poisson(x_val_clean, SEED+1) if POISSON_NOISE else x_val_clean
x_test = fixed_poisson(x_test_clean, SEED+2) if POISSON_NOISE else x_test_clean
a_val = a_raw[val_indices] * LRN_AD_INPUT_FACTOR
print('Train/validation/test:', len(x_train), len(x_val), len(x_test))
# Retain AD until the training cell builds labels using its configured threshold.
presence_ad_raw = a_raw
del x_raw, a_raw


Train/validation/test: 2760000 20000 20000


## 3. 128-wide attention, sparse presence, and frozen decoder
The decoder receives presence ? conditional magnitude. Reconstruction gradients pass through the entmax mapping wherever its support is active. The BCE classification loss provides corrective score gradients for incorrectly saturated absent/present predictions. Both losses participate from the first update; no BCE-only warm-up or Gaussian head.


In [6]:
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SPECTRUM_WIDTH = SPECTRUM_LENGTHS[METHOD]
DECODER_INPUT_SIZE = MAX_LAYERS * len(ELEMENTS)
# Fit log-count normalization on training spectra only, in bounded batches.
normalization_sum = np.zeros(SPECTRUM_WIDTH, dtype=np.float64)
normalization_sum_sq = np.zeros(SPECTRUM_WIDTH, dtype=np.float64)
for start in range(0, len(x_train), 1024):
    transformed = np.log1p(x_train[start:start+1024]).astype(np.float64)
    normalization_sum += transformed.sum(axis=0)
    normalization_sum_sq += np.square(transformed).sum(axis=0)
input_mean = (normalization_sum/len(x_train)).astype(np.float32)
input_variance = np.maximum(normalization_sum_sq/len(x_train)-np.square(normalization_sum/len(x_train)), 0)
input_std = np.maximum(np.sqrt(input_variance), 1e-3).astype(np.float32)
del normalization_sum, normalization_sum_sq, input_variance, transformed
lrn = LRN_AD(study, {METHOD: SPECTRUM_WIDTH}, elements=ELEMENTS,
    ad_input_factor=LRN_AD_INPUT_FACTOR, **LRN_AD_ARCHITECTURE).to(device)
lrn.load_state_dict(lrn_checkpoint['model_state_dict'])
lrn.eval().requires_grad_(False)
if not np.isclose(float(lrn.ad_input_factor.cpu()), LRN_AD_INPUT_FACTOR):
    raise ValueError('Decoder AD scaling differs from checkpoint')
loss_weights = torch.ones(SPECTRUM_WIDTH, device=device)
loss_weights[1100:] = 5.0
loss_weights /= loss_weights.mean()

def reconstruction_loss(predicted_scaled, clean_counts):
    targets = clean_counts * LRN_AD_OUTPUT_SCALE
    return (((predicted_scaled-targets).square()/(targets+1.0))*loss_weights).mean()



## 4. Training settings, training, and protected checkpoints
All editable encoder architecture, optimizer, loss, schedule, output and resume settings are at the top of the following training cell.


In [ ]:
# resume=True continues checkpoints in OUTPUT_DIR; resume=False requires a fresh directory.
OUTPUT_DIR = EXAMPLES / 'artifacts/encoder_attention_entmax_sparse_elements_lrn_ad'
PRESENCE_WEIGHT =1.0
PRESENCE_MIN_AD = 1  # Physical elemental AD per cell, in 1e15 atoms/cm2
ENCODER_ARCHITECTURE = dict(cnn_channels=(32,64,128,128),
    max_layers=MAX_LAYERS, elements_per_layer=len(lrn_checkpoint['elements']),
    attention_size=128, attention_heads=4, attention_blocks=2, dropout=0.0,
    kernel_size=7, initial_density=0.01)
BATCH_SIZE = 512
VALIDATION_BATCH_SIZE = 2048
EVALUATION_SAMPLES = 20_000
WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 1.0
resume = False  # True: load latest/best checkpoint; False: train from scratch in a fresh output directory.

# Graded learning-rate schedule
TRAINING_SCHEDULE = [
    {'learning_rate': 5e-4, 'epochs': 20},
    {'learning_rate': 1e-4, 'epochs': 30},
    {'learning_rate': 5e-5, 'epochs': 10},
]

# Refresh the model implementation when rerunning in an existing kernel.
import importlib
import ibeamlab.spectrum_encoder as encoder_module
importlib.reload(encoder_module)
SpectrumEntmaxEncoder = encoder_module.SpectrumEntmaxEncoder

# Build labels once; changing their threshold requires rerunning the data cell.
if 'presence_ad_raw' in globals():
    presence_raw = (presence_ad_raw > PRESENCE_MIN_AD).astype(np.float32)
    y_train, y_val, y_test = presence_raw[train_indices], presence_raw[val_indices], presence_raw[test_indices]
    presence_label_threshold = PRESENCE_MIN_AD
    del presence_raw, presence_ad_raw
elif 'presence_label_threshold' not in globals() or any(name not in globals() for name in ('y_train','y_val','y_test')):
    raise ValueError('Presence label cache is unavailable: rerun the data-loading and split cells, then this training cell')
elif presence_label_threshold != PRESENCE_MIN_AD:
    raise ValueError('Presence label threshold changed: rerun the data-loading and split cells first')

torch.manual_seed(SEED)
model = SpectrumEntmaxEncoder(SPECTRUM_WIDTH, DECODER_INPUT_SIZE,
    input_mean=input_mean, input_std=input_std,
    **ENCODER_ARCHITECTURE).to(device)
@torch.no_grad()
def evaluate(inputs, clean, batch_size=VALIDATION_BATCH_SIZE):
    model.eval(); lrn.eval()
    total = 0.0
    for start in range(0, len(inputs), batch_size):
        counts = torch.from_numpy(inputs[start:start+batch_size]).to(device)
        targets = torch.from_numpy(clean[start:start+batch_size]).to(device)
        total += float(reconstruction_loss(lrn(model(counts)), targets))*len(counts)
    return total / len(inputs)

# Check dimensions and differentiability before training; never update the decoder.
model.eval()
probe = torch.ones(2, SPECTRUM_WIDTH, device=device)
z = model(probe)
assert z.shape == (2, DECODER_INPUT_SIZE) and torch.isfinite(z).all() and (z >= 0).all()
reconstruction_loss(lrn(z), probe).backward()
assert model.output.weight.grad is not None and torch.isfinite(model.output.weight.grad).all()
assert all(p.grad is None for p in lrn.parameters())
model.zero_grad(set_to_none=True)
print(f'Training device: {device}; encoder output: {DECODER_INPUT_SIZE} scaled AD values')
print('Frozen decoder; scaled tail-weighted reconstruction + per-cell presence BCE')

if not math.isfinite(PRESENCE_WEIGHT) or PRESENCE_WEIGHT < 0 or not math.isfinite(PRESENCE_MIN_AD) or PRESENCE_MIN_AD < 0:
    raise ValueError('Invalid presence loss/label settings')
if not TRAINING_SCHEDULE or not isinstance(EVALUATION_SAMPLES, int) or EVALUATION_SAMPLES < 1:
    raise ValueError('Invalid schedule/evaluation size')
for stage in TRAINING_SCHEDULE:
    if (not isinstance(stage['epochs'], int) or isinstance(stage['epochs'], bool)
            or stage['epochs'] < 1 or not math.isfinite(stage['learning_rate']) or stage['learning_rate'] <= 0):
        raise ValueError('Use positive epochs and finite positive learning rates')
if BATCH_SIZE < 1 or VALIDATION_BATCH_SIZE < 1 or not math.isfinite(WEIGHT_DECAY) or WEIGHT_DECAY < 0 or not math.isfinite(MAX_GRAD_NORM) or MAX_GRAD_NORM <= 0:
    raise ValueError('Invalid optimizer/batch settings')
epoch_schedule = [(number, float(stage['learning_rate']))
                  for number, stage in enumerate(TRAINING_SCHEDULE, 1)
                  for _ in range(stage['epochs'])]
TOTAL_EPOCHS = len(epoch_schedule)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
weights_path = OUTPUT_DIR / 'encoder_entmax_best.pt'
resume_path = OUTPUT_DIR / 'encoder_entmax_resume.pt'
if not isinstance(resume, bool):
    raise ValueError('resume must be True or False')
if resume and not (resume_path.is_file() or weights_path.is_file()):
    raise FileNotFoundError('resume=True requires a latest/best checkpoint in OUTPUT_DIR; set resume=False explicitly to train from scratch')
if not resume and (resume_path.exists() or weights_path.exists()):
    raise ValueError('resume=False starts from scratch: choose a fresh OUTPUT_DIR to preserve existing checkpoints')

for output in (weights_path, resume_path):
    if output.resolve() in (LRN_AD_CHECKPOINT.resolve(),):
        raise ValueError('Encoder output must differ from source checkpoints')

def file_hash(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

signature = dict(model_type='SpectrumEntmaxEncoder', architecture=model.architecture,
    presence_weight=PRESENCE_WEIGHT, presence_min_ad=PRESENCE_MIN_AD,
    presence_mapping='independent_binary_entmax15', presence_loss='binary_cross_entropy_with_logits_score_difference_mean',
    config_sha256=file_hash(CONFIG_PATH), dataset_manifest=dataset_manifest,
    split_counts=(len(train_indices), len(val_indices), len(test_indices)),
    decoder_sha256=file_hash(LRN_AD_CHECKPOINT),
    input_size=SPECTRUM_WIDTH, output_size=DECODER_INPUT_SIZE,
    elements=list(ELEMENTS), max_layers=MAX_LAYERS,
    decoder_input_factor=LRN_AD_INPUT_FACTOR, output_scale=LRN_AD_OUTPUT_SCALE,
    loss='lrn_ad_scaled_tail_chi2_mean', tail_start=1100, tail_weight=5.0,
    poisson=POISSON_NOISE, seed=SEED, evaluation_samples=EVALUATION_SAMPLES)
resume_settings = dict(training_schedule=copy.deepcopy(TRAINING_SCHEDULE), batch_size=BATCH_SIZE,
    validation_batch_size=VALIDATION_BATCH_SIZE, weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM, noise_device_type=device.type,
    train_shape=tuple(x_train.shape), validation_shape=tuple(x_val.shape))

def fixed_subset(count, seed):
    if count <= EVALUATION_SAMPLES:
        return np.arange(count)
    return np.sort(np.random.default_rng(seed).choice(count, EVALUATION_SAMPLES, replace=False))

train_eval_indices = fixed_subset(len(x_train), SEED+101)
val_eval_indices = fixed_subset(len(x_val), SEED+102)
x_train_eval_clean = x_train[train_eval_indices].copy()
x_train_eval = fixed_poisson(x_train_eval_clean, SEED+103) if POISSON_NOISE else x_train_eval_clean
x_val_eval, x_val_eval_clean = x_val[val_eval_indices], x_val_clean[val_eval_indices]

def presence_bce(logits, targets):
    # Train the absent/present score difference with stable BCE; entmax still gates AD.
    return torch.nn.functional.binary_cross_entropy_with_logits(logits[...,1]-logits[...,0], targets)

def evaluate_validation():
    return evaluate(x_val_eval, x_val_eval_clean, VALIDATION_BATCH_SIZE)

@torch.inference_mode()
def evaluate_epoch_losses(inputs, clean, presence):
    model.eval(); lrn.eval()
    accumulated = torch.zeros(2, device=device, dtype=torch.float64)
    for start in range(0, len(inputs), VALIDATION_BATCH_SIZE):
        stop = start+VALIDATION_BATCH_SIZE
        counts = torch.from_numpy(inputs[start:stop]).to(device)
        targets = torch.from_numpy(clean[start:stop]).to(device)
        labels = torch.from_numpy(presence[start:stop]).to(device)
        predicted_ad, logits, probabilities = model.forward_components(counts)
        components = torch.stack((reconstruction_loss(lrn(predicted_ad), targets),
            presence_bce(logits, labels)))
        accumulated += components.double()*len(counts)
    reconstruction, presence_loss = (accumulated/len(inputs)).cpu().tolist()
    return dict(reconstruction=reconstruction, presence_bce=presence_loss,
        total=reconstruction+PRESENCE_WEIGHT*presence_loss)

# AD magnitudes provide a forward reference; binarized AD supplies per-cell presence labels.
with torch.inference_mode():
    reference_total = 0.0
    for start in range(0, len(val_eval_indices), VALIDATION_BATCH_SIZE):
        rows = val_eval_indices[start:start+VALIDATION_BATCH_SIZE]
        true_z = torch.from_numpy(a_val[rows]).to(device).flatten(1)
        clean = torch.from_numpy(x_val_clean[rows]).to(device)
        reference_total += float(reconstruction_loss(lrn(true_z), clean))*len(rows)
    true_profile_reference = reference_total/len(val_eval_indices)
print(f'True-profile validation reconstruction: {true_profile_reference:.6g}')
print(f'Fixed epoch evaluation: {len(train_eval_indices):,} train / {len(val_eval_indices):,} validation')

def snapshot_model():
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}

def snapshot_optimizer():
    result = copy.deepcopy(optimizer.state_dict())
    for state in result['state'].values():
        for name, value in state.items():
            if torch.is_tensor(value):
                state[name] = value.detach().cpu().clone()
    return result

def atomic_save(value, path):
    temporary = path.with_suffix('.pt.tmp')
    torch.save(value, temporary)
    temporary.replace(path)

def read_checkpoint(path):
    value = torch.load(path, map_location='cpu', weights_only=True)
    saved_signature, current_signature = dict(value['signature']), dict(signature)
    for settings_to_compare in (saved_signature, current_signature):
        settings_to_compare.pop('presence_weight', None)
        settings_to_compare.pop('element_count_weight', None)
        settings_to_compare.pop('element_count_penalty', None)
        settings_to_compare.pop('presence_loss', None)
    if saved_signature != current_signature:
        raise ValueError(f'Encoder/decoder/data settings differ: {path}')
    return value

optimizer = torch.optim.AdamW(model.parameters(), lr=epoch_schedule[0][1], weight_decay=WEIGHT_DECAY)
loader_generator = torch.Generator().manual_seed(SEED)
noise_generator = torch.Generator(device=device).manual_seed(SEED+3)
history, completed_epochs, best_history, best_epoch = [], 0, [], 0
best_score = evaluate_validation()
if not math.isfinite(best_score):
    raise FloatingPointError('Initial reconstruction is nonfinite; refusing to train')
best_state, best_optimizer_state = snapshot_model(), snapshot_optimizer()
previous = None
if resume and weights_path.exists():
    previous = read_checkpoint(weights_path)
    current = snapshot_model()
    model.load_state_dict(previous['model_state_dict'])
    score = evaluate_validation()
    if not math.isfinite(score) or not np.isclose(score, previous['best_validation_reconstruction'], rtol=1e-5, atol=1e-8):
        raise RuntimeError('Saved best reconstruction does not reproduce')
    if score <= best_score:
        best_score, best_state = score, snapshot_model()
        best_epoch, best_history = previous['best_epoch'], list(previous['history'])
        best_optimizer_state = previous['optimizer_state_dict']
    model.load_state_dict(current)

resume_checkpoint = read_checkpoint(resume_path) if resume and resume_path.exists() else None
if resume_checkpoint is not None:
    saved = resume_checkpoint['resume_settings']
    editable = {'training_schedule', 'batch_size', 'validation_batch_size', 'weight_decay', 'max_grad_norm'}
    if any(saved.get(key) != resume_settings.get(key) for key in set(saved)|set(resume_settings) if key not in editable):
        raise ValueError('Data/device settings differ from resume checkpoint')
    model.load_state_dict(resume_checkpoint['model_state_dict'])
    optimizer.load_state_dict(resume_checkpoint['optimizer_state_dict'])
    history = list(resume_checkpoint['history'])
    completed_epochs = resume_checkpoint['completed_epochs']
    loader_generator.set_state(resume_checkpoint['loader_rng_state'])
    noise_generator.set_state(resume_checkpoint['noise_rng_state'])
    torch.set_rng_state(resume_checkpoint['torch_rng_state'])
    if device.type == 'cuda':
        torch.cuda.set_rng_state(resume_checkpoint['cuda_rng_state'], device)
    score = evaluate_validation()
    if math.isfinite(score) and score < best_score:
        best_score, best_state = score, snapshot_model()
        best_optimizer_state = snapshot_optimizer()
        best_epoch, best_history = completed_epochs, list(history)
    print(f'Resuming encoder after {completed_epochs}/{TOTAL_EPOCHS} epochs')
else:
    model.load_state_dict(best_state)
    optimizer.load_state_dict(best_optimizer_state)
    if resume and previous is not None:
        history = list(best_history)
        completed_epochs = best_epoch
        print(f'Resuming protected best after {completed_epochs}/{TOTAL_EPOCHS} epochs (latest checkpoint unavailable)')
    else:
        best_history, best_epoch = [], 0
        print('Training from scratch (resume=False)')
for group in optimizer.param_groups:
    group['weight_decay'] = WEIGHT_DECAY
loader = DataLoader(TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)), batch_size=BATCH_SIZE,
                    shuffle=True, generator=loader_generator)

def checkpoint_metadata():
    return dict(signature=signature, true_profile_reference=true_profile_reference,
        train_indices=torch.from_numpy(train_indices.copy()), val_indices=torch.from_numpy(val_indices.copy()),
        test_indices=torch.from_numpy(test_indices.copy()),
        train_eval_indices=torch.from_numpy(train_eval_indices.copy()),
        val_eval_indices=torch.from_numpy(val_eval_indices.copy()))

def save_best():
    atomic_save(dict(**checkpoint_metadata(), model_state_dict=best_state,
        optimizer_state_dict=best_optimizer_state, history=copy.deepcopy(best_history),
        best_epoch=best_epoch, best_validation_reconstruction=best_score), weights_path)

def save_resume():
    atomic_save(dict(**checkpoint_metadata(), model_state_dict=snapshot_model(),
        optimizer_state_dict=snapshot_optimizer(), history=copy.deepcopy(history),
        completed_epochs=completed_epochs, resume_settings=resume_settings,
        loader_rng_state=loader_generator.get_state(), noise_rng_state=noise_generator.get_state(),
        torch_rng_state=torch.get_rng_state(),
        cuda_rng_state=torch.cuda.get_rng_state(device) if device.type == 'cuda' else None), resume_path)

def restore_best():
    protected = read_checkpoint(weights_path)
    model.load_state_dict(protected['model_state_dict'])
    model.to(device).eval(); lrn.eval()
    score = evaluate_validation()
    if not np.isclose(score, protected['best_validation_reconstruction'], rtol=1e-5, atol=1e-8):
        raise RuntimeError('Restored encoder best score differs from checkpoint')
    return protected

save_best()  # Protect baseline before the first update.
if resume_checkpoint is None:
    save_resume()
initial_validation = evaluate_epoch_losses(x_val_eval, x_val_eval_clean, y_val[val_eval_indices])
print(f"Initial validation loss BEFORE training: total={initial_validation['total']:.8g}, reconstruction={initial_validation['reconstruction']:.8g}")

try:
    with tqdm(total=TOTAL_EPOCHS, initial=completed_epochs, desc='Training epochs') as progress:
        for epoch in range(completed_epochs, TOTAL_EPOCHS):
            stage_number, learning_rate = epoch_schedule[epoch]
            for group in optimizer.param_groups:
                group['lr'] = learning_rate
            model.train(); lrn.eval()
            epoch_start = time.perf_counter()
            optimization_total, seen = 0.0, 0
            for clean, presence in loader:
                clean = clean.to(device)
                inputs = torch.poisson(clean, generator=noise_generator) if POISSON_NOISE else clean
                optimizer.zero_grad(set_to_none=True)
                predicted_ad, presence_logits, presence_probabilities = model.forward_components(inputs)
                loss = (reconstruction_loss(lrn(predicted_ad), clean)
                        + PRESENCE_WEIGHT*presence_bce(presence_logits, presence.to(device)))
                if not torch.isfinite(loss):
                    raise FloatingPointError('Nonfinite reconstruction loss')
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM, error_if_nonfinite=True)
                optimizer.step()
                optimization_total += float(loss.detach())*len(clean)
                seen += len(clean)
            train_finished = time.perf_counter()
            train_metrics = evaluate_epoch_losses(x_train_eval, x_train_eval_clean, y_train[train_eval_indices])
            validation_metrics = evaluate_epoch_losses(x_val_eval, x_val_eval_clean, y_val[val_eval_indices])
            train_score = train_metrics['reconstruction']
            validation_score = validation_metrics['reconstruction']
            if not math.isfinite(train_metrics['total']) or not math.isfinite(validation_metrics['total']):
                raise FloatingPointError('Nonfinite total epoch loss')
            if not math.isfinite(train_score) or not math.isfinite(validation_score):
                raise FloatingPointError('Nonfinite epoch evaluation')
            history.append(dict(epoch=epoch+1, stage=stage_number, learning_rate=learning_rate,
                train_reconstruction=train_score, val_reconstruction=validation_score,
                train_total=train_metrics['total'], val_total=validation_metrics['total'],
                train_loss_components=train_metrics, val_loss_components=validation_metrics,
                optimization_loss=optimization_total/seen,
                training_seconds=train_finished-epoch_start,
                evaluation_seconds=time.perf_counter()-train_finished))
            completed_epochs = epoch+1
            if validation_score < best_score:
                best_score, best_state = validation_score, snapshot_model()
                best_optimizer_state = snapshot_optimizer()
                best_epoch, best_history = completed_epochs, copy.deepcopy(history)
                save_best()
            save_resume()
            progress.set_postfix(dict(best_rec=f'{best_score:.6g}', lr=f'{learning_rate:.1g}',
                stage=stage_number, train_rec=f'{train_score:.6g}', val_rec=f'{validation_score:.6g}',
                train_total=f"{train_metrics['total']:.6g}", val_total=f"{validation_metrics['total']:.6g}"), refresh=False)
            progress.update(1)
except KeyboardInterrupt:
    print('Interrupted; partial epoch replays from the latest completed checkpoint')
finally:
    model.load_state_dict(best_state)
    model.to(device).eval(); lrn.eval()
protected = restore_best()
print(f"Protected best epoch {protected['best_epoch']}: validation reconstruction {protected['best_validation_reconstruction']:.6g}")
print('Latest completed-epoch resume:', resume_path)


d:\ibeamlab\ibeamlab\spectrum_encoder.py:193: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attended = F.scaled_dot_product_attention(q, k, v,


Training device: cuda; encoder output: 100 scaled AD values
Frozen decoder; scaled tail-weighted reconstruction + per-cell presence BCE
True-profile validation reconstruction: 0.00287889
Fixed epoch evaluation: 20,000 train / 20,000 validation
Training from scratch (resume=False)
Initial validation loss BEFORE training: total=78.210216, reconstruction=77.513991


Training epochs:  30%|███       | 18/60 [3:08:52<7:09:20, 613.34s/it, best_rec=0.0479099, lr=0.0005, stage=1, train_rec=0.0605312, val_rec=0.0619146, train_total=0.300417, val_total=0.30435] 

## 5. Held-out evaluation and reconstruction examples

Always restore protected best weights. The encoder output is a reconstruction coordinate, not a calibrated physical profile or uncertainty estimate.


In [ ]:
protected = restore_best()
test_reconstruction = evaluate(x_test, x_test_clean, VALIDATION_BATCH_SIZE)
print(f'Held-out test reconstruction: {test_reconstruction:.6g}')
report = dict(best_epoch=protected['best_epoch'], best_validation_reconstruction=protected['best_validation_reconstruction'],
    test_reconstruction=test_reconstruction, true_profile_reference=true_profile_reference,
    completed_epochs=completed_epochs, signature=signature)
(OUTPUT_DIR/'encoder_metrics.json').write_text(json.dumps(report, indent=2)+'\n', encoding='utf8')
if history:
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.plot([r['epoch'] for r in history], [r['train_reconstruction'] for r in history], label='fixed train')
    ax.plot([r['epoch'] for r in history], [r['val_reconstruction'] for r in history], label='validation')
    ax.axhline(true_profile_reference, linestyle='--', label='true-profile reference')
    ax.set(xlabel='Epoch', ylabel='LRN_AD reconstruction loss'); ax.legend(); plt.show()
preview_count = min(16, len(x_test))
with torch.inference_mode():
    preview = lrn(model(torch.from_numpy(x_test[:preview_count]).to(device))).cpu().numpy()/LRN_AD_OUTPUT_SCALE
fig, axes = plt.subplots(4, 4, figsize=(20, 12), constrained_layout=True)
for row, ax in enumerate(axes.flat):
    if row >= preview_count:
        ax.set_visible(False)
        continue
    ax.plot(x_test_clean[row], label='clean target', linewidth=1)
    ax.plot(preview[row], label='encoder + frozen LRN_AD', linewidth=1)
    ax.set(title=f'Test sample {row + 1}', xlabel='Channel', ylabel='Counts')
    ax.legend(fontsize=8)
plt.show()

@torch.inference_mode()
def presence_metrics(inputs, targets):
    probabilities = []
    model.eval()
    for start in range(0, len(inputs), VALIDATION_BATCH_SIZE):
        _, _, p = model.forward_components(torch.from_numpy(inputs[start:start+VALIDATION_BATCH_SIZE]).to(device))
        probabilities.append(p.cpu().numpy())
    probabilities = np.concatenate(probabilities)
    truth = targets.astype(bool)
    decisions = probabilities >= 0.5
    metrics = {}
    for j, element in enumerate(ELEMENTS):
        true = truth[:,:,j]; pred = decisions[:,:,j]
        tp = int((true & pred).sum()); fp = int((~true & pred).sum()); fn = int((true & ~pred).sum())
        metrics[element] = dict(precision=tp/max(1,tp+fp), recall=tp/max(1,tp+fn),
            false_positive_rate=fp/max(1,int((~true).sum())),
            brier=float(np.mean((probabilities[:,:,j]-targets[:,:,j])**2)),
            exact_zero_fraction=float((probabilities[:,:,j]==0).mean()))
    sample_probability = 1 - np.prod(1-probabilities,axis=1)
    sample_truth = truth.any(axis=1)
    sample_decisions = sample_probability >= 0.5
    sample_tp = sample_decisions & sample_truth
    sample_fp = sample_decisions & ~sample_truth
    sample_fn = ~sample_decisions & sample_truth
    sample_metrics = {}
    for j, element in enumerate(ELEMENTS):
        tp, fp, fn = int(sample_tp[:,j].sum()), int(sample_fp[:,j].sum()), int(sample_fn[:,j].sum())
        sample_metrics[element] = dict(precision=tp/max(1,tp+fp), recall=tp/max(1,tp+fn),
            false_positives=fp, false_negatives=fn,
            false_positive_rate=fp/max(1,int((~sample_truth[:,j]).sum())))
    return dict(per_cell=metrics, per_sample_element=sample_metrics,
        mean_predicted_elements=float(sample_decisions.sum(1).mean()),
        mean_true_elements=float(sample_truth.sum(1).mean()),
        mean_expected_elements=float(sample_probability.sum(1).mean()),
        mean_false_positive_elements=float(sample_fp.sum(1).mean()),
        mean_false_negative_elements=float(sample_fn.sum(1).mean()))
presence_report = dict(validation=presence_metrics(x_val,y_val), test=presence_metrics(x_test,y_test))
(OUTPUT_DIR/'presence_metrics.json').write_text(json.dumps(presence_report,indent=2)+'\n',encoding='utf8')
print('Held-out presence and element-count metrics:', presence_report['test'])


## 6. Export protected best encoder as ZIP
The package contains the gated encoder only, with spectrum normalization embedded and output scaling to physical elemental AD. It loads through the usual Python/native C++ inverse-model API. LRN_AD remains a training decoder and is not bundled. This encoder has no learned uncertainty or presence output.

The ZIP returns gated AD. The PyTorch checkpoint also exposes per-cell presence probabilities via `forward_components`; the ZIP does not declare predicted uncertainty.


In [ ]:
protected = restore_best()
# Load the protected state into the current export implementation without restarting the kernel.
importlib.reload(encoder_module)
export_model = encoder_module.SpectrumEntmaxEncoder(SPECTRUM_WIDTH, DECODER_INPUT_SIZE,
    **protected['signature']['architecture']).cpu().eval()
export_model.load_state_dict(protected['model_state_dict'])
package_path = OUTPUT_DIR / f'encoder_attention_entmax_{MAX_LAYERS}layers.zip'
export_suffix = 1
while package_path.exists():
    package_path = OUTPUT_DIR / f'encoder_attention_entmax_{MAX_LAYERS}layers_{export_suffix:03d}.zip'
    export_suffix += 1
export_model.export(package_path, study=study, method=METHOD, elements=ELEMENTS,
    ad_input_factor=LRN_AD_INPUT_FACTOR, need_pileup_subtraction=True)
native_model = ibl.InverseModel(package_path)
check_inputs = np.array(x_test[:min(16, len(x_test))], dtype=np.float32, copy=True)
with torch.inference_mode():
    reference = torch.cat([export_model(torch.from_numpy(row[None])) for row in check_inputs]).numpy()
measurements = [{METHOD: ibl.Spectrum(METHOD, row)} for row in check_inputs]
predictions = native_model.predict_prepared(measurements)
native_ad = np.stack([result.values for result in predictions])
np.testing.assert_allclose(native_ad.reshape(len(check_inputs), -1) * LRN_AD_INPUT_FACTOR,
    reference, rtol=2e-4, atol=1e-6, err_msg='Native encoder ZIP differs from protected CPU PyTorch weights')
assert native_model.metadata.class_name == 'SpectrumEntmaxEncoder'
assert not any(result.uncertainty_predicted for result in predictions)
print(f"Exported protected best epoch {protected['best_epoch']}; validation reconstruction {protected['best_validation_reconstruction']:.6g}")
print('Verified native/PyTorch AD parity:', package_path)
